# MiBici Guadalajara — Raw Ingestion to Staging

This notebook loads the raw monthly trip CSVs and the station metadata CSV into a `staging` schema in PostgreSQL, untouched. No cleaning or transformation happens here, that's handled in a separate notebook (`02_cleaning.ipynb`). The goal of this staging layer is to preserve an auditable, unmodified copy of the source data.


## 1. Setup: connect to PostgreSQL

Credentials are loaded from a local `.env` file (not committed to version control) so the connection string never contains plaintext secrets.

In [26]:
# Import and load .env credentials
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

Quick sanity check. Confirms the engine can actually reach the server before we try loading any data.

In [5]:
# Connection test
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("SELECT version();"))
    print(result.fetchone())

('PostgreSQL 16.3, compiled by Visual C++ build 1939, 64-bit',)


## 2. Load trip data into staging

A reusable function infers the table name (`trips_YYYY_MM`) from the filename and loads the CSV directly into the `staging` schema. This same function will be used for all 12 months.

Reading directly from `data/raw/`, no parsing/cleaning applied yet,this is intentionally the rawest possible read, matching the source file exactle.

In [7]:
import re

def load_trip_csv_to_staging(filename, engine, raw_dir="../data/raw/"):
    match = re.search(r"(\d{4})_(\d{2})", filename)
    if not match:
        raise ValueError(f"Couldn't find a YYYY_MM pattern in filename: {filename}")
    year, month = match.groups()
    table_name = f"trips_{year}_{month}"

    df = pd.read_csv(f"{raw_dir}{filename}", encoding="latin1")
    df.to_sql(table_name, con=engine, schema="staging", if_exists="replace", index=False)

    print(f"Loaded {df.shape[0]} rows into staging.{table_name}")
    return df

In [9]:
trips_jan = load_trip_csv_to_staging("datos_abiertos_2025_01.csv", engine)
print(trips_jan.shape)
trips_jan.head()

Loaded 380184 rows into staging.trips_2025_01
(380184, 8)


,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,37162342,601273,M,1982.0,2025-01-01 00:00:44,2025-01-01 00:11:29,211,395
1,37162343,1571524,M,2003.0,2025-01-01 00:04:11,2025-01-01 00:15:07,35,260
2,37162344,1435200,F,1984.0,2025-01-01 00:05:27,2025-01-01 00:15:23,35,260
3,37162345,126327,M,1988.0,2025-01-01 00:07:47,2025-01-01 00:17:42,190,11
4,37162346,416121,M,1994.0,2025-01-01 00:08:52,2025-01-01 00:18:10,273,8


## 3. Load station metadata into staging

Station data is a single file (not monthly), so it gets its own simple load. No need for the filename-parsing function used for trips.

In [19]:
stations = pd.read_csv(
    "../data/raw/nomenclatura_2026.csv",
    encoding="utf-8"
)

print(stations.shape)
stations.head()

(484, 5)


,id,name,latitude,longitude,dpcapacity
0,2,(GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.,20.666378,-103.348820,15
1,3,(GDL-002) C. Colonias / Av. Niños héroes,20.667228,-103.366000,15
2,4,(GDL-003) C. Vidrio / Av. Chapultepec,20.667690,-103.368252,19
3,5,(GDL-004) C. Ghilardi /C. Miraflores,20.691847,-103.362549,19
4,6,(GDL-005) C. San Diego /Calzada Independencia,20.681158,-103.339363,11


Loaded as-is into `staging.stations`. Note: this file is UTF-8 encoded, unlike the monthly trip CSVs (Latin-1/cp1252).

In [13]:
stations.to_sql(
    "stations",
    con=engine,
    schema="staging",
    if_exists="replace",
    index=False
)

print(f"Loaded {stations.shape[0]} rows into staging.stations")

Loaded 484 rows into staging.stations


## 5. Scale to all 12 months

Updated the loader to try UTF-8 first, falling back to Latin-1 if decoding fails, since the station file was UTF-8 but January's trip file was Latin-1, encoding is evidently inconsistent across files from this source. The encoding actually used is logged per file for documentation purposes.

In [15]:
def load_trip_csv_to_staging(filename, engine, raw_dir="../data/raw/"):
    match = re.search(r"(\d{4})_(\d{2})", filename)
    if not match:
        raise ValueError(f"Couldn't find a YYYY_MM pattern in filename: {filename}")
    year, month = match.groups()
    table_name = f"trips_{year}_{month}"

    try:
        df = pd.read_csv(f"{raw_dir}{filename}", encoding="utf-8")
        used_encoding = "utf-8"
    except UnicodeDecodeError:
        df = pd.read_csv(f"{raw_dir}{filename}", encoding="latin1")
        used_encoding = "latin1"

    df.to_sql(table_name, con=engine, schema="staging", if_exists="replace", index=False)
    print(f"Loaded {df.shape[0]} rows into staging.{table_name} (encoding: {used_encoding})")
    return df

Loop over every trip CSV in `data/raw/` matching the naming pattern and load each into its own monthly staging table.

In [18]:
import glob

trip_files = sorted(glob.glob("../data/raw/datos_abiertos_2025_*.csv"))

for file_path in trip_files:
    filename = os.path.basename(file_path)
    load_trip_csv_to_staging(filename, engine)

Loaded 380184 rows into staging.trips_2025_01 (encoding: latin1)
Loaded 372529 rows into staging.trips_2025_02 (encoding: latin1)
Loaded 405110 rows into staging.trips_2025_03 (encoding: utf-8)
Loaded 366193 rows into staging.trips_2025_04 (encoding: utf-8)
Loaded 395891 rows into staging.trips_2025_05 (encoding: utf-8)
Loaded 356611 rows into staging.trips_2025_06 (encoding: utf-8)
Loaded 368186 rows into staging.trips_2025_07 (encoding: latin1)
Loaded 374727 rows into staging.trips_2025_08 (encoding: utf-8)
Loaded 376789 rows into staging.trips_2025_09 (encoding: utf-8)
Loaded 420064 rows into staging.trips_2025_10 (encoding: utf-8)
Loaded 380328 rows into staging.trips_2025_11 (encoding: utf-8)
Loaded 335420 rows into staging.trips_2025_12 (encoding: utf-8)
